# YOLO26x - Active Learning Fine-Tune

Bu notebook `active_learning_training.zip` içindeki insan-düzeltilmiş active learning karelerini COCO formatından YOLO formatına çevirir ve `yolo26x.pt` modelini 50 epoch fine-tune eder.

`data.yaml` 80 COCO sınıf adıyla yazılır ve `single_cls` **kullanılmaz**. Sebebi: `nc=1` verildiğinde ultralytics sınıf kafasının son konvolüsyonlarını şekil uyuşmazlığından yükleyemiyor (`1009/1015` tensör transfer oluyor) ve COCO'nun person sınıflandırıcısını rastgele değerlerle değiştiriyor. Model "insan nedir"i 397 kutudan sıfırdan öğrenmeye çalıştığı için 20 epoch sonunda bile eğitilmemiş modelin altında kalıyordu.

Ayarlar:

- `epochs=50`, `save_period=10` yani her 10 epochta checkpoint
- `device=0,1` yani Kaggle T4 x2
- `imgsz=1536`
- `nbs=batch` çünkü varsayılan `nbs=64` bu veri boyutunda epoch başına ~1 optimizer adımı bırakıyor
- `optimizer=AdamW lr0=0.0002` çünkü `optimizer=auto` verilen `lr0`'ı yok sayıp 0.002 kullanıyor ve eğitim ıraksıyor
- `freeze=10` yani backbone donuk; ultralytics donuk katmanların BatchNorm'unu her epoch `eval()`'da tutuyor, running stats bozulmuyor
- `TILE = False` ilk koşuda. `True` yapınca kareler çıkarımdaki döşeme geometrisiyle kesilir, eğitim seti 73 → ~291 kareye çıkar ve `imgsz=1536`'da medyan kutu kenarı 25.5 px yerine 65.7 px olur. İki değişikliğin katkısını ayırmak için ikinci koşuya bırakıldı.

Değerlendirme uyarısı: aşağıdaki `model.val()` çıktısı 20 karelik / 48 kutuluk val split'i ölçer ve bu split train'den belirgin şekilde daha zordur (medyan kutu 15.7 px vs 27.0 px, kutuların %50'si 16 px altında). Eğitilmemiş `yolo11x` bu split'te `imgsz=1536`'da AP50 = 0.224 ve conf 0.01'de max recall 0.25 alıyor. Yani buradaki mAP hem gürültülü hem karamsar; kararı `evaluate_on_gt.py` ile dokunulmamış 100 karelik GT test setinden ver.

Kaggle'da ayrı hesapta bu notebook'u aç, `active_learning_training.zip` ve `yolo26x.pt` dosyasını input olarak ekle, Accelerator olarak **GPU T4 x2** seçip Save Version / Run All çalıştır.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocotools"], check=True)

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import YAML

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU sayisi:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
else:
    raise SystemExit("GPU yok. Kaggle Settings > Accelerator > GPU T4 x2 secin.")

In [ ]:
from collections import defaultdict

import cv2

ACTIVE_ZIP_NAME = "active_learning_training.zip"
RAW_DIR = Path("/kaggle/working/active_learning_raw")
YOLO_DIR = Path("/kaggle/working/active_learning_yolo")
VAL_VIDEO_PREFIX = "DJI_0574"  # video-level validation split

# TILE=True olunca kareler cikarimdaki doseme geometrisiyle kesilir: egitim seti 73 -> ~291
# kareye cikar ve imgsz=1536'da medyan kutu kenari 25.5 px yerine 65.7 px olur.
# Ilk kosuda False: once nc duzeltmesinin tek basina ne kazandirdigini gormek istiyoruz,
# yoksa iki degisikligin katkisi birbirine karisir. Ikinci kosuda True yapip karsilastir.
TILE = False
TILE_OVERLAP = 0.2
MIN_BOX_FRAC = 0.4  # kutunun bu kadari doseme icinde kalirsa etiket olarak yazilir
EMPTY_KEEP = 5      # kutusuz dosemelerin 5'te 1'i negatif ornek olarak tutulur


def auto_grid(width, height):
    """tiled_dino.auto_grid ile ayni."""
    long_side = max(width, height)
    if long_side >= 3000:
        return 3, 3
    if long_side >= 1200:
        return 2, 2
    return 1, 1


def tile_windows(width, height, rows, cols, overlap=TILE_OVERLAP):
    """tiled_dino.tile_windows ile ayni."""
    if rows == 1 and cols == 1:
        return [(0, 0, width, height)]
    tile_w = min(width, int(round(width / cols * (1 + overlap))))
    tile_h = min(height, int(round(height / rows * (1 + overlap))))
    step_x = (width - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (height - tile_h) / (rows - 1) if rows > 1 else 0
    windows = []
    for r in range(rows):
        for c in range(cols):
            x1, y1 = int(round(c * step_x)), int(round(r * step_y))
            windows.append((x1, y1, x1 + tile_w, y1 + tile_h))
    return windows


def find_file(root, name):
    for dirpath, _, filenames in os.walk(root):
        if name in filenames:
            return Path(dirpath) / name
    return None


zip_path = find_file("/kaggle/input", ACTIVE_ZIP_NAME)
if zip_path is None:
    raise SystemExit(
        f"{ACTIVE_ZIP_NAME} bulunamadi. Add Input ile elle etiketledigin active learning zip'ini ekle."
    )
print("Training zip:", zip_path)

shutil.rmtree(RAW_DIR, ignore_errors=True)
shutil.rmtree(YOLO_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
YOLO_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(RAW_DIR)

ann_path = RAW_DIR / "annotations" / "instances_default.json"
if not ann_path.exists():
    raise SystemExit(f"COCO annotation bulunamadi: {ann_path}")

coco = json.loads(ann_path.read_text())
images = {im["id"]: im for im in coco["images"]}
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        continue
    anns_by_image[ann["image_id"]].append(ann)

for split in ("train", "val"):
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def find_image(file_name):
    # CVAT COCO export file_name sadece basename tutuyor; gorseller zip icinde images/default altinda.
    direct = RAW_DIR / file_name
    if direct.exists():
        return direct
    matches = list(RAW_DIR.rglob(Path(file_name).name))
    if not matches:
        raise FileNotFoundError(file_name)
    return matches[0]


split_counts = defaultdict(lambda: {"images": 0, "boxes": 0, "empty": 0})
empty_seen = 0
for image_id, im in sorted(images.items()):
    stem = Path(im["file_name"]).stem
    split = "val" if stem.rsplit("_f", 1)[0] == VAL_VIDEO_PREFIX else "train"
    width, height = im["width"], im["height"]
    frame = cv2.imread(str(find_image(Path(im["file_name"]).name)))
    if frame is None:
        raise SystemExit(f"Gorsel okunamadi: {stem}")
    boxes = [a["bbox"] for a in anns_by_image.get(image_id, []) if a["bbox"][2] > 0 and a["bbox"][3] > 0]

    windows = tile_windows(width, height, *auto_grid(width, height)) if TILE else [(0, 0, width, height)]
    for idx, (x1, y1, x2, y2) in enumerate(windows):
        tile_w, tile_h = x2 - x1, y2 - y1
        lines = []
        for bx, by, bw, bh in boxes:
            ix1, iy1 = max(bx, x1), max(by, y1)
            ix2, iy2 = min(bx + bw, x2), min(by + bh, y2)
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            if inter / (bw * bh) < MIN_BOX_FRAC:
                continue
            lines.append(
                "0 %.6f %.6f %.6f %.6f"
                % (
                    ((ix1 + ix2) / 2 - x1) / tile_w,
                    ((iy1 + iy2) / 2 - y1) / tile_h,
                    (ix2 - ix1) / tile_w,
                    (iy2 - iy1) / tile_h,
                )
            )
        if not lines:
            empty_seen += 1
            # Alt orneklemeyi sadece dosemede yap: 308 bos doseme 229 pozitifi bastirirdi.
            # Tam karede toplam 3 bos kare var, hepsi negatif ornek olarak degerli.
            if TILE and empty_seen % EMPTY_KEEP:
                continue
            split_counts[split]["empty"] += 1
        name = f"{stem}_t{idx}" if TILE else stem
        cv2.imwrite(
            str(YOLO_DIR / "images" / split / f"{name}.jpg"),
            frame[y1:y2, x1:x2],
            [cv2.IMWRITE_JPEG_QUALITY, 95],
        )
        (YOLO_DIR / "labels" / split / f"{name}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        split_counts[split]["images"] += 1
        split_counts[split]["boxes"] += len(lines)

if split_counts["val"]["images"] == 0:
    raise SystemExit(f"Val split bos: {VAL_VIDEO_PREFIX} veri icinde yok, VAL_VIDEO_PREFIX'i degistir.")

# nc=1 verirsek ultralytics sinif kafasinin son konvolusyonlarini (model.23.cv3.*.2) sekil
# uyusmazligi yuzunden yukleyemiyor ve rastgele baslatiyor: 1009/1015 tensor transfer oluyor,
# COCO'nun ~250k insan ornegiyle egitilmis person siniflandiricisi cope gidiyor. 80 COCO adini
# yazip etiketleri sinif 0'da tutunca 1015/1015 transfer oluyor. Diger 79 sinif hic pozitif
# ornek gormuyor, cikarimda da classes=[0] ile filtreliyoruz.
coco_names = YAML.load(Path(ultralytics.__file__).parent / "cfg" / "datasets" / "coco.yaml")["names"]
data_yaml = YOLO_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {YOLO_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in sorted(coco_names.items()))
)

print("\nYOLO dataset hazir:", YOLO_DIR, "| TILE =", TILE)
for split in ("train", "val"):
    c = split_counts[split]
    print(f"  {split:<5}: {c['images']:>4} image ({c['empty']} negatif), {c['boxes']:>4} box")
print(f"\ndata.yaml: nc={len(coco_names)} (COCO adlari korundu, etiketlerin hepsi sinif 0)")

In [ ]:
MODEL_FILE = "yolo26x.pt"
RUN_NAME = "yolo26x_active_50ep"
EPOCHS = 50
IMGSZ = 1536
BATCH = 4        # toplam batch. OOM alirsan BATCH ve NBS'i birlikte 2 yap.
NBS = BATCH      # accumulate = round(NBS / BATCH) = 1
DEVICE = "0,1"   # Kaggle T4 x2
WORKERS = 2

model_path = find_file("/kaggle/input", MODEL_FILE) or MODEL_FILE
print("Model:", model_path)

# Varsayilanlarla ilk deneme 3. epochta iraksadi (cls_loss 12 -> 1961, mAP 0). Iki sebep vardi:
#   1) nbs=64 / batch=4 -> accumulate=16, yani epoch basina ~1 optimizer adimi.
#   2) optimizer=auto, verilen lr0'i yok sayip AdamW lr=0.002 seciyor (fine-tune icin ~10 kat fazla).
cmd = [
    "yolo", "detect", "train",
    f"model={model_path}",
    f"data={data_yaml}",
    f"epochs={EPOCHS}",
    f"imgsz={IMGSZ}",
    f"batch={BATCH}",
    f"nbs={NBS}",
    f"device={DEVICE}",
    f"workers={WORKERS}",
    "optimizer=AdamW",
    "lr0=0.0002",
    "lrf=0.05",
    "warmup_epochs=1",
    "freeze=10",     # backbone donuk; donuk katmanlarin BN'i ultralytics tarafindan eval'da tutulur
    "mosaic=0.0",    # kucuk nesneleri daha da kucultuyor
    "scale=0.5",
    "fliplr=0.5",
    # single_cls=True BURAYA KONMAZ: data.yaml ne derse desin nc'yi 1'e zorluyor
    # (trainer.py: 'if self.args.single_cls: data["nc"] = 1') ve pretrained sinif kafasini bozar.
    "amp=True",
    "cache=False",
    "save=True",
    "save_period=10",
    "patience=30",
    "project=/kaggle/working/runs_yolo_active",
    f"name={RUN_NAME}",
    "exist_ok=True",
]
print(" ".join(str(x) for x in cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Eğitim bittikten sonra val split üzerinde son bir ölçüm.
best = Path(f"/kaggle/working/runs_yolo_active/{RUN_NAME}/weights/best.pt")
last = Path(f"/kaggle/working/runs_yolo_active/{RUN_NAME}/weights/last.pt")
model = YOLO(str(best if best.exists() else last))
metrics = model.val(data=str(data_yaml), imgsz=IMGSZ, device=DEVICE, plots=True)
print(metrics)
print("\nCheckpoint klasoru:", f"/kaggle/working/runs_yolo_active/{RUN_NAME}/weights")

## Çıktılar

Checkpointler burada olur:

`/kaggle/working/runs_yolo_active/<run_name>/weights/`

Özellikle:

- `best.pt`
- `last.pt`
- `epoch10.pt`, `epoch20.pt`, `epoch30.pt`, `epoch40.pt`, `epoch50.pt`

Sonraki adımda bu checkpointleri aynı `evaluate_on_gt.py` mantığıyla 100 karelik dokunulmamış test GT üzerinde ölçeceğiz.